# 📋 CPZÃO · Notebook 01 — Aquisição e Preparação do Dataset

**Auditoria de Acessibilidade em Calçadas** · Detecção de objetos YOLOv8

Este notebook monta o dataset do zero: baixa as anotações de uma base pública aberta,
converte para o formato YOLO, seleciona um subconjunto balanceado, divide em
train/valid/test e gera as estatísticas que entram no README.

**Não é preciso capturar nem anotar nenhuma imagem.** Rode tudo de cima para baixo.

---

### Fonte dos dados

**Image Dataset of Accessibility Barriers**
Stolberg, Jakob (2022) — Alexander von Humboldt Institute for Internet and Society
🔗 https://zenodo.org/records/6382090 · **Licença CC-BY-4.0**

5.538 fotos de espaços públicos anotadas com caixas delimitadoras, com um
[protocolo de anotação documentado e rigoroso](https://zenodo.org/records/6382090)
— é essa documentação que sustenta a consistência da rotulagem.

### Classes

| id | classe | origem | o que é |
|:--:|:--|:--|:--|
| 0 | `degrau` | `step` | desnível isolado: guia de calçada, soleira, mudança de nível |
| 1 | `escada` | `stair` | dois ou mais degraus em sequência |
| 2 | `corrimao` | `grab_bar` | apoio/guarda-corpo de escada ou rampa |

> A quarta label do dataset original, `ramp`, foi **descartada de propósito**: tem 143
> instâncias contra 3.564 de `step` (1:25). Com desbalanceamento desse tamanho a classe
> não aprende e ainda contamina a matriz de confusão das outras.

### O truque que evita baixar 8 GB

O dataset original é um zip único de **8 GB**. Mas as anotações são um XML de 192 KB
comprimido, lá no começo do arquivo. Usando **range requests HTTP** (`Range: bytes=...`),
este notebook lê o índice do zip remotamente, extrai só o XML, decide quais imagens
interessam — e **baixa apenas essas**. Em vez de 8 GB, baixa ~400 MB.

### Ordem de execução

`01_dataset_prep.ipynb` (este) → `02_treino_export.ipynb`

> 🖥️ **Runtime:** `Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)`

## 1. Ambiente e código do projeto

In [ ]:
!pip install -q ultralytics

# Clona o repositório do projeto — traz o módulo que monta o dataset
# (scripts/zenodo_dataset.py) e o app de inferência.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/viniciusborgess/cpzao-auditoria-calcadas.git"
REPO = Path("/content/cpzao-auditoria-calcadas")

if not REPO.exists():
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q

sys.path.insert(0, str(REPO / "scripts"))
import zenodo_dataset as z

print("✅ módulo carregado")
print("   classes :", z.CLASSES)
print("   fonte   :", z.ZENODO_URL.split("/files/")[0])

In [ ]:
import collections, json, random, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "❌ nenhuma (ative a GPU no runtime antes do notebook 02)")

## 2. Configuração

`ALVO` é quantas imagens o dataset final terá. A rubrica exige **no mínimo 60**.

- **600** (padrão) — ~5 min de download, treina em ~12 min. Melhor resultado.
- **200** — ~2 min de download. Suficiente e bem acima do mínimo.
- **80** — download quase instantâneo, para testar o pipeline rápido.

In [ ]:
ALVO     = 600      # imagens no dataset final
WORKERS  = 6        # downloads simultâneos (o Zenodo devolve 429 acima disso)
MAX_LADO = 640      # redimensiona as imagens; o treino roda em 192
SEED     = 42

DEST  = Path("/content/dataset_calcadas")
IMGSZ = 192         # travado pelo app (app/src/inference.js: INPUT_W/H = 192)

CLASSES   = z.CLASSES
NUM_CLASS = len(CLASSES)

print(f"alvo    : {ALVO} imagens")
print(f"classes : {CLASSES}")
print(f"imgsz   : {IMGSZ} (travado pelo app mobile)")

## 3. Ler o índice do zip remoto

Nenhum byte de imagem é baixado aqui — só o diretório central do zip (~517 KB), que diz
onde cada arquivo está dentro dos 8 GB.

In [ ]:
t0 = time.perf_counter()
entries = z.read_central_directory()
por_nome = {e["name"]: e for e in entries}

imgs_zip = [e for e in entries if e["name"].lower().endswith(".jpg")]
tam_total = sum(e["csize"] for e in entries) / 1e9

print(f"{len(entries)} entradas lidas em {time.perf_counter()-t0:.1f}s")
print(f"  imagens no zip : {len(imgs_zip)}")
print(f"  tamanho total  : {tam_total:.2f} GB  ← é isso que NÃO vamos baixar")
print(f"  anotações      : {por_nome['wm_annotations.xml']['csize']/1e3:.0f} KB comprimido")

## 4. Extrair e converter as anotações (CVAT XML → YOLO)

In [ ]:
xml = z.extract_member(z.ZENODO_URL, por_nome["wm_annotations.xml"])
print(f"XML: {len(xml)/1e6:.2f} MB descomprimido")

registros = z.parse_annotations(xml)
st_orig = z.estatisticas(registros)

print(f"\n── dataset original ──")
print(f"  imagens            : {st_orig['imagens']}")
print(f"  com objeto anotado : {st_orig['com_caixa']}")
print(f"  sem objeto         : {st_orig['negativas']}")
print(f"  caixas (3 classes) : {st_orig['caixas']}")
for c, n in st_orig["por_classe"].items():
    print(f"    {c:<10s} {n:>5d}")

### Como a conversão funciona

O CVAT guarda a caixa em **pixels absolutos** (`xtl`, `ytl`, `xbr`, `ybr`) junto com a
largura e altura da imagem. O YOLO quer **centro e tamanho normalizados** em `[0,1]`:

```
cx = (xtl + xbr) / 2 / W        w = (xbr - xtl) / W
cy = (ytl + ybr) / 2 / H        h = (ybr - ytl) / H
```

Antes de normalizar, `parse_annotations` recorta a caixa aos limites da imagem e descarta
caixas degeneradas (largura ou altura zero), que gerariam `NaN` na loss.

## 5. Seleção balanceada

Pegar imagens aleatórias não serve: `degrau` aparece em quase toda foto e `corrimao` em
poucas, então uma amostra aleatória ficaria dominada por `degrau`.

A estratégia é processar as classes **da mais rara para a mais comum**. Como `corrimao` e
`escada` quase sempre aparecem acompanhados de `degrau`, priorizar as raras eleva as três
ao mesmo tempo.

Também entram ~6% de **imagens negativas** (sem nenhum objeto). Elas ensinam o modelo a
*não* detectar — é o que reduz falso positivo na demo.

In [ ]:
sel = z.select_balanced(registros, alvo=ALVO, frac_negativas=0.06, seed=SEED)
st_sel = z.estatisticas(sel)

print(f"selecionadas: {st_sel['imagens']} imagens | {st_sel['caixas']} caixas")
print(f"  com objeto : {st_sel['com_caixa']}")
print(f"  negativas  : {st_sel['negativas']}")
for c, n in st_sel["por_classe"].items():
    print(f"    {c:<10s} {n:>5d}")

bal = min(st_sel["por_classe"].values()) / max(st_sel["por_classe"].values())
print(f"\nbalanceamento (menor/maior): {bal:.2f}")
print("  → acima de 0.5 é saudável;", "✅ ok" if bal >= 0.5 else "⚠️ desbalanceado")

## 6. Baixar as imagens selecionadas

Só as imagens escolhidas, via range requests paralelos.

> ⏱️ Com `ALVO=600` leva ~5 min. O Zenodo limita requisições por minuto e devolve
> **HTTP 429** quando se passa do limite — `_http_range` já trata isso com retry e backoff
> exponencial. Não aumente `WORKERS`: testamos 16 e perdeu ~60% dos downloads.

In [ ]:
alvos = [por_nome[r["name"]] for r in sel]

t0 = time.perf_counter()
def prog(n, t):
    el = time.perf_counter() - t0
    print(f"  {n}/{t} ({n/t*100:.0f}%) — {el/60:.1f} min, ETA {el/n*(t-n)/60:.1f} min",
          flush=True)

blobs = z.extract_many(z.ZENODO_URL, alvos, workers=WORKERS, progresso=prog)

dt = time.perf_counter() - t0
mb = sum(len(v) for v in blobs.values()) / 1e6
print(f"\n✅ {len(blobs)}/{len(alvos)} imagens | {mb:.0f} MB em {dt/60:.1f} min "
      f"({mb/dt:.1f} MB/s)")

# descarta registros cuja imagem não veio
sel = [r for r in sel if r["name"] in blobs]
print(f"   {len(sel)} registros utilizáveis")

### Plano B — baixar o zip inteiro

Se o rate limit do Zenodo estiver agressivo e muitos downloads falharem, baixar os 8 GB de
uma vez é **mais** robusto: é uma requisição só, então não existe limite a estourar. No
Colab a banda é boa e isso leva poucos minutos.

In [ ]:
USAR_ZIP_COMPLETO = False   # mude para True se a célula anterior perdeu muitas imagens

if USAR_ZIP_COMPLETO:
    def prog_zip(feito, total):
        if feito % (200 << 20) < (1 << 20):
            print(f"  {feito/1e9:.2f} / {total/1e9:.2f} GB", flush=True)

    zf = z.baixar_zip_completo("/content/wm_barriers_data.zip", progresso=prog_zip)
    sel = z.select_balanced(registros, alvo=ALVO, frac_negativas=0.06, seed=SEED)
    blobs = {r["name"]: zf.read(r["name"]) for r in sel}
    print(f"✅ {len(blobs)} imagens lidas do zip local")
else:
    print("pulado (a célula anterior resolveu)")

## 7. Escrever o dataset no formato YOLO

Divisão **70/20/10** em train/valid/test, como a rubrica exige.

As imagens são reduzidas para 640 no maior lado (originais são ~1350×1800) e o **EXIF é
removido**. Remover o EXIF não é cosmético: são fotos de celular e podem trazer a tag de
orientação; como as anotações foram feitas sobre as dimensões armazenadas no arquivo,
qualquer rotação automática na leitura desalinharia todas as caixas.

In [ ]:
import shutil
shutil.rmtree(DEST, ignore_errors=True)

splits = z.split_registros(sel, seed=SEED)
z.escrever_dataset(splits, DEST, lambda n: blobs[n], max_lado=MAX_LADO)

DATA_YAML = str(DEST / "data.yaml")
por_split = {s: z.estatisticas(r) for s, r in splits.items()}
ordem = ["train", "valid", "test"]

for s in ordem:
    e = por_split[s]
    print(f"{s:<6s} {e['imagens']:4d} img  {e['caixas']:4d} caixas  {e['por_classe']}")

disco = sum(f.stat().st_size for f in DEST.rglob('*.jpg')) / 1e6
print(f"\ndisco: {disco:.0f} MB")
print(f"\n── data.yaml ──")
print(Path(DATA_YAML).read_text())

## 8. Validação das anotações

Varre os `.txt` escritos procurando os problemas que silenciosamente arruínam um treino:

| problema | efeito |
|:--|:--|
| `class_id` fora de `[0, nc-1]` | o treino aborta com *IndexError* |
| coordenada fora de `[0,1]` | caixa cortada ou ignorada |
| largura/altura = 0 | caixa degenerada, vira `NaN` na loss |
| caixa menor que 2 px | some ao reduzir para 192; o modelo aprende ruído |
| imagem sem label | tratada como fundo puro — derruba o recall |

In [ ]:
problemas = []

for s in ordem:
    idir, ldir = DEST / s / "images", DEST / s / "labels"
    for ip in sorted(idir.glob("*.jpg")):
        lp = ldir / (ip.stem + ".txt")
        if not lp.exists():
            problemas.append(f"[{s}] {ip.name}: sem arquivo de label")
            continue

        W, H = Image.open(ip).size
        for n, linha in enumerate(lp.read_text().splitlines(), 1):
            if not linha.strip():
                continue
            p = linha.split()
            if len(p) != 5:
                problemas.append(f"[{s}] {lp.name}:{n}: {len(p)} campos (esperado 5)")
                continue
            cid = int(p[0]); cx, cy, bw, bh = map(float, p[1:])
            if not (0 <= cid < NUM_CLASS):
                problemas.append(f"[{s}] {lp.name}:{n}: class_id {cid} fora de [0,{NUM_CLASS-1}]")
            if not all(0 <= v <= 1 for v in (cx, cy, bw, bh)):
                problemas.append(f"[{s}] {lp.name}:{n}: coords fora de [0,1]")
            if bw <= 0 or bh <= 0:
                problemas.append(f"[{s}] {lp.name}:{n}: caixa degenerada")
            elif bw * W < 2 or bh * H < 2:
                problemas.append(f"[{s}] {lp.name}:{n}: caixa menor que 2px")

print("═" * 58)
if problemas:
    print(f"🚨 {len(problemas)} problema(s):\n")
    for p in problemas[:25]:
        print("  •", p)
    if len(problemas) > 25:
        print(f"  ... e mais {len(problemas)-25}")
else:
    print("✅ nenhum problema estrutural nas anotações")
print("═" * 58)

## 9. Conformidade com a rubrica

In [ ]:
tot_img = sum(por_split[s]["imagens"] for s in ordem)
tot_cx  = sum(por_split[s]["caixas"]  for s in ordem)
tot_cls = {c: sum(por_split[s]["por_classe"][c] for s in ordem) for c in CLASSES}

checks = [
    ("≥ 60 imagens rotuladas",         tot_img >= 60,                            f"{tot_img} imagens"),
    ("≥ 3 classes distintas",          NUM_CLASS >= 3,                           f"{NUM_CLASS} classes"),
    ("todas as classes com instância", all(v > 0 for v in tot_cls.values()),     str(tot_cls)),
    ("≥ 20 instâncias por classe",     all(v >= 20 for v in tot_cls.values()),   str(tot_cls)),
    ("split train presente",           por_split["train"]["imagens"] > 0,        f"{por_split['train']['imagens']} img"),
    ("split valid presente",           por_split["valid"]["imagens"] > 0,        f"{por_split['valid']['imagens']} img"),
    ("split test presente",            por_split["test"]["imagens"]  > 0,        f"{por_split['test']['imagens']} img"),
    ("sem erros estruturais",          len(problemas) == 0,                      f"{len(problemas)} erros"),
]

print("═" * 58)
print("CONFORMIDADE COM A RUBRICA — Dataset & Rotulagem (4,0 pts)")
print("═" * 58)
for nome, ok, det in checks:
    print(f"  {'✅' if ok else '❌'}  {nome:<30s} {det}")
print("═" * 58)
print("🎉 dataset aprovado — siga para o notebook 02" if all(o for _, o, _ in checks)
      else "⚠️  resolva os itens ❌ antes de treinar")

## 10. Estatísticas e gráficos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
cores = {"train": "#2563eb", "valid": "#f59e0b", "test": "#10b981"}

# (a) instâncias por classe, empilhado por split
base = np.zeros(NUM_CLASS)
for s in ordem:
    v = np.array([por_split[s]["por_classe"][c] for c in CLASSES], float)
    axes[0].bar(CLASSES, v, 0.6, bottom=base, label=s, color=cores[s])
    base += v
for i in range(NUM_CLASS):
    axes[0].text(i, base[i] + max(base) * .02, int(base[i]), ha="center", fontweight="bold")
axes[0].set_title("Instâncias por classe", fontweight="bold")
axes[0].set_ylabel("nº de caixas"); axes[0].legend()

# (b) imagens por split
v = [por_split[s]["imagens"] for s in ordem]
b = axes[1].bar(ordem, v, color=[cores[s] for s in ordem])
axes[1].bar_label(b, fmt="%d", fontweight="bold")
axes[1].set_title(f"Imagens por split (total {tot_img})", fontweight="bold")
axes[1].set_ylabel("nº de imagens")

# (c) tamanho das caixas
areas = [bw * bh for r in sel for _, _, _, bw, bh in r["boxes"]]
axes[2].hist(np.sqrt(areas) * 100, bins=30, color="#6366f1", edgecolor="white")
axes[2].axvline(np.sqrt(areas).mean() * 100, color="#dc2626", ls="--",
                label=f"média {np.sqrt(areas).mean()*100:.0f}%")
axes[2].set_title("Tamanho das caixas", fontweight="bold")
axes[2].set_xlabel("√área (% do lado da imagem)"); axes[2].legend()

plt.tight_layout()
plt.savefig("dataset_stats.png", dpi=130, bbox_inches="tight")
plt.show()

print(f"média de caixas por imagem: {tot_cx/tot_img:.2f}")
print(f"balanceamento: {min(tot_cls.values())/max(tot_cls.values()):.2f}")

## 11. Inspeção visual das anotações

**Olhe de verdade para esta grade.** É a forma mais rápida de confirmar que a conversão
CVAT→YOLO está correta: se as caixas estiverem deslocadas, rotacionadas ou com a classe
errada, aparece aqui.

In [ ]:
PALETA = {0: "#ff1744", 1: "#ffd600", 2: "#00e676"}   # mesmas cores do app

mostra = sorted([r for r in splits["train"] if len(r["boxes"]) >= 2],
                key=lambda r: -len(r["boxes"]))[:9]
if len(mostra) < 9:
    mostra += [r for r in splits["train"] if r not in mostra][:9 - len(mostra)]

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for ax, r in zip(axes.ravel(), mostra):
    ip = DEST / "train" / "images" / (Path(r["name"]).stem + ".jpg")
    im = Image.open(ip).convert("RGB")
    W, H = im.size
    dr = ImageDraw.Draw(im)
    for cid, cx, cy, bw, bh in r["boxes"]:
        x1, y1 = (cx - bw/2) * W, (cy - bh/2) * H
        x2, y2 = (cx + bw/2) * W, (cy + bh/2) * H
        cor = PALETA[cid]
        dr.rectangle([x1, y1, x2, y2], outline=cor, width=4)
        rot = CLASSES[cid]
        dr.rectangle([x1, max(0, y1-18), x1 + 9*len(rot), y1], fill=cor)
        dr.text((x1+3, max(0, y1-17)), rot, fill="black")
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{len(r['boxes'])} caixa(s)", fontsize=9)
for ax in axes.ravel()[len(mostra):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("anotacoes_amostra.png", dpi=110, bbox_inches="tight")
plt.show()

## 12. Exportar as estatísticas para o README

In [ ]:
L = ["| split | imagens | caixas | " + " | ".join(f"`{c}`" for c in CLASSES) + " |",
     "|:--|--:|--:|" + "--:|" * NUM_CLASS]
for s in ordem:
    e = por_split[s]
    L.append(f"| **{s}** | {e['imagens']} | {e['caixas']} | "
             + " | ".join(str(e["por_classe"][c]) for c in CLASSES) + " |")
L.append(f"| **TOTAL** | **{tot_img}** | **{tot_cx}** | "
         + " | ".join(f"**{tot_cls[c]}**" for c in CLASSES) + " |")
tabela = "\n".join(L)

resumo = f'''## 📊 Estatísticas do Dataset

{tabela}

- **Total de imagens:** {tot_img}
- **Total de caixas:** {tot_cx}
- **Média de caixas por imagem:** {tot_cx/tot_img:.2f}
- **Imagens negativas (sem objeto):** {sum(por_split[s]["negativas"] for s in ordem)} — reduzem falso positivo
- **Divisão:** {por_split["train"]["imagens"]} train / {por_split["valid"]["imagens"]} valid / {por_split["test"]["imagens"]} test (70/20/10)
- **Balanceamento entre classes:** {min(tot_cls.values())/max(tot_cls.values()):.2f} (menor/maior)
- **Resolução:** reduzida para {MAX_LADO} no maior lado, EXIF removido
- **Validação das labels:** {len(problemas)} problema(s) estrutural(is)
- **Fonte:** [Image Dataset of Accessibility Barriers](https://zenodo.org/records/6382090) — Stolberg, Jakob (2022), CC-BY-4.0
'''

print(resumo)
Path("dataset_stats.md").write_text(resumo)
json.dump({"total_imagens": tot_img, "total_caixas": tot_cx, "por_classe": tot_cls,
           "por_split": {s: por_split[s] for s in ordem}, "classes": CLASSES,
           "imgsz": IMGSZ, "problemas_labels": len(problemas)},
          open("dataset_stats.json", "w"), indent=2, ensure_ascii=False)
print("✅ gerados: dataset_stats.md · dataset_stats.json · dataset_stats.png · anotacoes_amostra.png")

In [ ]:
# Baixa os artefatos para commitar em docs/metricas/
from google.colab import files
for f in ["dataset_stats.md", "dataset_stats.json",
          "dataset_stats.png", "anotacoes_amostra.png"]:
    try:
        files.download(f)
    except Exception as e:
        print("não baixou", f, e)

## 13. Passar o dataset para o notebook 02

Se você rodar os dois notebooks na **mesma sessão** do Colab, as variáveis abaixo são
recuperadas automaticamente pelo notebook 02.

In [ ]:
print("DATA_YAML =", repr(DATA_YAML))
%store DATA_YAML
%store CLASSES
%store IMGSZ

---

## ✅ Próximo passo

Dataset montado e aprovado na rubrica → abra **[`02_treino_export.ipynb`](02_treino_export.ipynb)**.

Commite em `docs/metricas/` os arquivos baixados: `dataset_stats.png` e
`anotacoes_amostra.png` são evidência direta de rotulagem consistente na correção.

> 💡 **Quer melhorar a nota da demo?** Este dataset é de espaços públicos europeus.
> Degrau, escada e corrimão são visualmente universais, então o modelo transfere bem —
> mas se sobrar tempo, acrescentar 20 fotos do próprio corredor da faculdade
> (ver [`CAPTURA.md`](../CAPTURA.md)) ancora o modelo no domínio exato da apresentação.